# Sapsucker Woods: 25% versus 30% evaluation budget

**Kenneth Mwangi Kabiru · University of Leeds**

This is a **post hoc budget sensitivity analysis** on the existing five species and 100 acquisition histories. Training remains at 3,000 labels; recall must remain at least 90%, with a 5% single-attempt false-acceptance bound. **The question is whether LR mean correct acceptance exceeds 80% at a 30% label budget.** The three other fixed rankers are secondary.

Read **EXPERIMENT.md** in this folder for the full design. The original 25% study remains unchanged. This notebook compares two paired counterfactual audit policies, not a second chance after a failed audit.

**CPU is sufficient.** This notebook has no audio-download or Perch-inference stage. It verifies the original Drive cache, or restores the same arrays from your original results ZIP.

- **saved_histories** (fast): reuse original purchased labels and final model scores; rerun ranking, audits and analysis.
- **rebuild_from_embeddings**: refit the original acquisition histories and classifiers from cached embeddings, then run the same analysis. This costs CPU time but does not require the original results ZIP when the complete cache is available.

Both routes must match the original 25% results before the 30% stage becomes available. All new outputs stay in a separate Drive folder.


In [ ]:
# @title Configuration {vertical-output: true}
PROJECT_DIRECTORY = "/content/drive/MyDrive/ssw2" # @param {type: 'string'}
DRIVE_CACHE_PATH = "/content/drive/MyDrive/ssw_cache" # @param {type: 'string'}
RUN_MODE = "saved_histories" # @param ["saved_histories", "rebuild_from_embeddings"]
RESULTS_PATH = "" # @param {type: 'string'}
OUTPUT_DIRECTORY = "/content/drive/MyDrive/ssw2_outputs" # @param {type: 'string'}


## 1. Mount Drive and locate this folder

Upload the complete `ssw2` folder, including `helpers`. `RESULTS_PATH` may be empty when the original results ZIP is still in `DRIVE_CACHE_PATH/result_exports`. Otherwise enter its full Drive path. It may also point to an unpacked original `ssw_results` folder. In refit mode the verified complete cache is sufficient.

Input storage is read-only. The output directory must be separate from the project, original cache and results inputs.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
paths=[PROJECT_DIRECTORY,DRIVE_CACHE_PATH,RESULTS_PATH,OUTPUT_DIRECTORY]
if any(str(p).startswith('/content/drive/') for p in paths):
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT=Path(PROJECT_DIRECTORY).expanduser().resolve()
if not (PROJECT/'project.json').is_file():
    raise RuntimeError('PROJECT_DIRECTORY must point to the uploaded ssw2 folder containing project.json and helpers.')
if RUN_MODE not in ['saved_histories','rebuild_from_embeddings']:raise ValueError('Invalid run mode')
OUTPUT=Path(OUTPUT_DIRECTORY).expanduser().resolve()
for protected in [PROJECT,Path(DRIVE_CACHE_PATH).expanduser().resolve()]+([Path(RESULTS_PATH).expanduser().resolve()] if RESULTS_PATH.strip() else []):
    if OUTPUT==protected or protected in OUTPUT.parents or OUTPUT in protected.parents:
        raise RuntimeError('OUTPUT_DIRECTORY must be separate from the project, input cache and results packet.')
spec=json.loads((PROJECT/'project.json').read_text())
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
release=digest(PROJECT/'project.json')[:16]
RUN_ROOT=(Path('/content/ssw2_runs') if Path('/content').exists() else PROJECT.parent/'ssw2_runs')/release/RUN_MODE
RUN_ROOT.mkdir(parents=True,exist_ok=True)
for name in spec['static_files']+spec['helper_files']:
    source=PROJECT/('helpers/'+name if name in spec['helper_files'] else name)
    if digest(source)!=spec['execution_files'][name]:raise RuntimeError('Project file checksum mismatch: '+name)
    shutil.copyfile(source,RUN_ROOT/name)
shutil.copyfile(PROJECT/'project.json',RUN_ROOT/'project.json')
request={'drive_cache_path':str(Path(DRIVE_CACHE_PATH).expanduser().resolve()),'results_path':RESULTS_PATH.strip(),
         'output_directory':str(OUTPUT),'mode':RUN_MODE}
session=RUN_ROOT/'session.json'
if session.exists() and json.loads(session.read_text())!=request:
    raise RuntimeError('Paths changed in an existing local run. Restart the Colab session with the intended configuration.')
session.write_text(json.dumps(request,indent=2)+'\n')
os.chdir(RUN_ROOT)
print('PROJECT_READY; original inputs remain read-only')


## 2. Original model fitting and acquisition

This visible code retains the original LR/SVM parameters, pool fit order, paid startup and entropy acquisition. It executes only in `rebuild_from_embeddings` mode. Each completed LR fit is checkpointed. The fast route loads arrays instead.


In [ ]:
%%writefile learning.py
"""Raw-LR acquisition; only purchased labels enter fitting or selection."""
from support import np, core, array_hash
import math, random, time, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def training_arrays(ids,observed):
    idx=np.asarray([i for i,w in enumerate(ids) if w in observed],dtype=np.int64)
    y=np.asarray([observed[ids[i]] for i in idx],dtype=np.int64)
    if set(y.tolist())!={0,1}:raise ValueError('Two classes required')
    return idx,y

def native_predictions(decision):
    a=np.asarray(decision,dtype=np.float64)
    if not np.isfinite(a).all():raise ValueError('Nonfinite decision')
    return (a>0).astype(np.int8)

def fit_lr(X,ids,observed,seed):
    started=time.monotonic();idx,y=training_arrays(ids,observed)
    model=LogisticRegression(C=1,solver='lbfgs',penalty='l2',class_weight='balanced',fit_intercept=True,
                             tol=1e-4,max_iter=200,random_state=seed)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always');model.fit(X[idx],y)
    if model.classes_.tolist()!=[0,1]:raise RuntimeError('Invalid LR classes')
    p=np.asarray(model.predict_proba(X)[:,1],dtype=np.float64)
    decision=np.asarray(model.decision_function(X),dtype=np.float64)
    pred=native_predictions(decision)
    assert np.array_equal(pred,model.predict(X))
    if not np.isfinite(p).all():raise RuntimeError('Nonfinite LR probability')
    meta={'model':'raw_lr','count':len(idx),'positive_training':int(y.sum()),'fit_order':'pool',
          'training_indices_sha256':array_hash(idx),'training_labels_sha256':array_hash(y),
          'score_sha256':array_hash(p),'n_iter':model.n_iter_.tolist(),
          'warnings':[str(w.message) for w in caught],'seconds':time.monotonic()-started}
    return {'probability':p,'decision':decision,'prediction':pred,'coef':model.coef_.copy(),
            'intercept':model.intercept_.copy()},meta

def fit_svm(X,ids,observed,seed):
    started=time.monotonic();idx,y=training_arrays(ids,observed)
    scaler=StandardScaler().fit(X[idx])
    model=SVC(C=1,kernel='rbf',gamma='scale',class_weight='balanced',probability=False,tol=1e-3,
              shrinking=True,max_iter=-1,cache_size=200,random_state=seed)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always');model.fit(scaler.transform(X[idx]),y)
    if model.classes_.tolist()!=[0,1] or model.fit_status_!=0:raise RuntimeError('Invalid SVM fit')
    scores=np.asarray(model.decision_function(scaler.transform(X)),dtype=np.float64)
    if not np.isfinite(scores).all():raise RuntimeError('Nonfinite SVM scores')
    arrays={'scores':scores,'scaler_mean':scaler.mean_,'scaler_var':scaler.var_,'scaler_scale':scaler.scale_,
            'support_vectors':model.support_vectors_,'support':model.support_,'dual_coef':model.dual_coef_,
            'intercept':model.intercept_}
    meta={'model':'standardized_rbf_svm','count':len(idx),'positive_training':int(y.sum()),'fit_order':'pool',
          'training_indices_sha256':array_hash(idx),'training_labels_sha256':array_hash(y),
          'score_sha256':array_hash(scores),'classes':model.classes_.tolist(),'gamma':float(model._gamma),
          'n_support':model.n_support_.tolist(),'n_iter':model.n_iter_.tolist(),'scaler_fit_count':int(scaler.n_samples_seen_),
          'warnings':[str(w.message) for w in caught],'seconds':time.monotonic()-started,
          'array_sha256':{k:array_hash(v) for k,v in arrays.items()}}
    return arrays,meta

def entropy_query(scores,ids,observed,k):
    def entropy(i):
        p=min(1-1e-12,max(1e-12,float(scores[i])))
        return -p*math.log(p)-(1-p)*math.log(1-p)
    unseen=[i for i,w in enumerate(ids) if w not in observed]
    return sorted(unseen,key=lambda i:(-entropy(i),ids[i]))[:k]

def acquire(ids,X,oracle,seed,horizon=3000,batch=500,checkpoint=None,resume=None):
    """A completed snapshot is the only resumable acquisition boundary."""
    fits=[];snapshots=[];queries=[]
    if resume:
        order=list(map(int,resume['order']));startup=int(resume['startup'])
        snapshots=resume['snapshots'];fits=resume['fits'];queries=resume['queries']
        for j,i in enumerate(order):oracle.purchase(ids[i],'paid_search_startup' if j<startup else 'training')
        if resume.get('status')=='STARTUP_UNAVAILABLE':
            return {'order':order,'startup':startup,'snapshots':[],'fits':[],'queries':[], 'status':'STARTUP_UNAVAILABLE'}
    else:
        shuffled=list(range(len(ids)));random.Random(seed).shuffle(shuffled)
        order=[];classes=set()
        for i in shuffled[:horizon]:
            classes.add(oracle.purchase(ids[i],'paid_search_startup'));order.append(i)
            if len(classes)==2:break
        startup=len(order)
        if len(classes)<2:
            return {'order':order,'startup':startup,'snapshots':[],'fits':[],'queries':[], 'status':'STARTUP_UNAVAILABLE'}
    while True:
        if not snapshots or int(snapshots[-1]['count'])!=len(order):
            arrays,meta=fit_lr(X,ids,oracle.observed,seed)
            arrays['count']=len(order);snapshots.append(arrays);fits.append(meta)
            state={'order':order.copy(),'startup':startup,'snapshots':snapshots,'fits':fits,'queries':queries,'status':'CANDIDATE'}
            if checkpoint:checkpoint(state)
            print('LR_FIT',seed,len(order),round(meta['seconds'],2),flush=True)
        if len(order)==horizon:break
        selected=entropy_query(snapshots[-1]['probability'],ids,oracle.observed,min(batch,horizon-len(order)))
        if not selected or len(set(selected))!=len(selected):raise RuntimeError('Invalid acquisition batch')
        queries.append({'before':len(order),'indices':selected,'score_sha256':array_hash(snapshots[-1]['probability'])})
        for i in selected:
            if ids[i] in oracle.observed:raise RuntimeError('Duplicate acquisition')
            oracle.purchase(ids[i],'training');order.append(i)
    return {'order':order,'startup':startup,'snapshots':snapshots,'fits':fits,'queries':queries,'status':'CANDIDATE'}


## 3. Candidate freezing, paid census and a single audit

The same four fixed rankers and the original audit allocator are used at both budgets. The allocator sees paid census counts, never hidden tail labels. Evaluator-only labels calculate outcome probabilities after the candidate and audit are fixed. The longer exact arithmetic lives in the three neighbouring audit helpers; an independent combinatorial check also verifies every trace.


In [ ]:
%%writefile evaluate.py
"""Evaluate a fixed candidate; design precedes evaluator-only metrics."""
from support import *
from dataclasses import asdict
METHODS={"lr":"LR-RAW-FA","svm":"SVM-STD-FA","equal_blend":"LR50-SVM50-FA","3to1_blend":"LR75-SVM25-FA"}

def evaluate_candidate(ids,truth,order,startup,lr,svm,workflow,seed,stop_status,cap=5889,flagged_count=2355):
    # The only object constructed from outcomes before design is a paid oracle.
    candidate=core.freeze_candidate(ids,lr,svm,METHODS[workflow['ranker']],flagged_count)
    purchased=[ids[int(i)] for i in order];oracle=core.PaidOracle(truth,cap)
    for j,w in enumerate(purchased):oracle.purchase(w,'paid_search_startup' if j<startup else 'training')
    U,L,F=set(ids),set(purchased),set(candidate.flagged_ids)
    audit_order=core.common_order(ids,seed,991230000)
    trace=core.OneAttempt().run(candidate,purchased,oracle,audit_order)
    trace.update(seed=seed,workflow=workflow['id'])
    pre=len(L|F);D=U-(L|F)
    row={'seed':seed,'workflow':workflow['id'],'ranker':workflow['ranker'],'stop':workflow['stop'],
         'stop_status':stop_status,'training_count':len(order),'startup_paid':startup,'candidate_available':True,
         'flagged_count':len(F),'overlap':len(L&F),'pre_attention':pre,'flagged_id_hash':candidate.flagged_id_hash,
         'purchased_history_hash':core.canonical_hash([[w,truth[w]] for w in purchased]),
         'realized_total_attention':trace['actual_total_attention'],'realized_accepted':trace['terminal_state']=='ACCEPT',
         'terminal_reason':trace['terminal_reason'],'N':len(ids),'attention_cap':cap}
    # All evaluator-only access follows the frozen candidate and audit attempt.
    M=int(sum(truth[w] for w in D));T=int(sum(truth[w] for w in F));b=int(sum(truth[w] for w in L-F))
    if pre>cap:
        total=T+b+M;adequate=bool(total>0 and 10*T>=9*total)
        row.update(design=None,design_status='CENSUS_UNAFFORDABLE',boundary_check=None,
                   evaluator_only_M=M,evaluator_only_T=T,evaluator_only_b=b,
                   evaluator_only_recall=T/total if total else 0.,evaluator_only_adequate=adequate,
                   conditional_accept_probability=0.,correct_accept_probability=0.,false_accept_probability=0.,
                   expected_audit_inspected=0.,expected_total_attention=len(order),planned_total_attention=len(order))
    else:
        design=core.plan_audit(d=int(trace['unseen_d']),T=int(trace['T']),b_known=int(trace['b_known']),pre_attention=pre,cap=cap)
        row.update(design=asdict(design),design_status=design.status,boundary_check=core.verify_design(design))
        row.update(core.retrospective_metrics(design,evaluator_only_M=M))
        row.update(evaluator_only_T=T,evaluator_only_b=b)
    row['unaccepted_adequacy']=float(row['evaluator_only_adequate'])*(1-row['conditional_accept_probability'])
    row['realized_correct_accept']=row['realized_accepted'] and row['evaluator_only_adequate']
    row['realized_false_accept']=row['realized_accepted'] and not row['evaluator_only_adequate']
    assert row['planned_total_attention']<=cap and row['realized_total_attention']<=cap
    return row,trace,candidate.flagged_ids

def unavailable_row(ids,order,startup,workflow,seed,cap=5889):
    row={'seed':seed,'workflow':workflow['id'],'ranker':workflow['ranker'],'stop':workflow['stop'],
         'stop_status':'STARTUP_UNAVAILABLE','training_count':len(order),'startup_paid':startup,'candidate_available':False,
         'design':None,'design_status':'STARTUP_UNAVAILABLE','boundary_check':None,
         'evaluator_only_recall':None,'evaluator_only_adequate':None,'conditional_accept_probability':0.,
         'correct_accept_probability':0.,'false_accept_probability':0.,'unaccepted_adequacy':0.,
         'expected_total_attention':len(order),'planned_total_attention':len(order),'realized_total_attention':len(order),
         'realized_accepted':False,'realized_correct_accept':False,'realized_false_accept':False,'N':len(ids),'attention_cap':cap}
    trace={'seed':seed,'workflow':workflow['id'],'training_charged_ids':[ids[i] for i in order],
           'census_charged_ids':[],'audit_charged_ids':[],'audit_observed_labels':[],'observed_hits':0,
           'actual_total_attention':len(order),'audit_inspected':0,'terminal_state':'STOP_UNACCEPTED','terminal_reason':'STARTUP_UNAVAILABLE',
           'ledger':[{'window_id':ids[i],'purpose':'paid_search_startup','charge':1} for i in order]}
    return row,trace


## 4. Pair the budgets and require baseline reconciliation

The baseline stage verifies every original purchased-label and score hash, rebuilds flagged sets and reproduces all 400 fixed-method 25% rows. The 30% stage is a separate call and refuses to run before the complete baseline gate passes. Each budget has its own ledger and one audit attempt, sharing the original label-independent audit ordering.


In [ ]:
%%writefile budget_experiment.py
"""Two fixed budgets, with a complete baseline gate before any 30% evaluation."""
from support import *
from inputs import verified_arrays,find_packet
from evaluate import evaluate_candidate
from checks import check
import argparse

def history_key(target,seed):return target+'/'+str(seed)

def verify_history(pack,reference,ids,labels,baseline):
    order=pack['order'].astype(np.int64);lr=pack['lr'];svm=pack['svm']
    assert order.shape==(3000,) and len(set(order.tolist()))==3000 and order.min()>=0 and order.max()<len(ids)
    assert int(pack['startup'])==reference['startup']
    assert lr.shape==svm.shape==(len(ids),) and np.isfinite(lr).all() and np.isfinite(svm).all()
    if array_hash(lr)!=reference['lr_score_sha256'] or array_hash(svm)!=reference['svm_score_sha256']:
        raise RuntimeError('Rebuilt/saved scores do not match the original histories; 30% analysis is blocked')
    identity=core.canonical_hash([[ids[i],int(labels[i])] for i in order])
    assert all(identity==r['purchased_history_hash'] for r in baseline)
    for w in load(ROOT/'experiment.json')['workflows']:
        from evaluate import METHODS
        candidate=core.freeze_candidate(ids,lr,svm,METHODS[w['ranker']],2205)
        assert candidate.flagged_id_hash==next(r for r in baseline if r['workflow']==w['id'])['flagged_id_hash']

def save_progress(folder,state):
    arrays={'order':np.asarray(state['order'],dtype=np.int64),'startup':np.asarray(state['startup'])}
    for j,snapshot in enumerate(state['snapshots']):
        for key,value in snapshot.items():arrays['snapshot_%d_%s'%(j,key)]=np.asarray(value)
    staged=folder/'acquisition.pending.npz';savez(staged,**arrays)
    digest=sha(staged);filename='acquisition_'+digest+'.npz';os.replace(staged,folder/filename)
    dump(folder/'acquisition.json',{'file':filename,'file_sha256':digest,
         'snapshot_count':len(state['snapshots']), 'fits':state['fits'],'queries':state['queries'],'status':state['status']})

def read_progress(folder):
    if not (folder/'acquisition.json').exists():return None
    meta=load(folder/'acquisition.json')
    filename=meta.pop('file')
    if Path(filename).name!=filename:raise RuntimeError('Unsafe acquisition checkpoint name')
    if sha(folder/filename)!=meta.pop('file_sha256'):raise RuntimeError('Acquisition checkpoint checksum mismatch')
    arrays=loadz(folder/filename);snapshots=[]
    for j in range(meta.pop('snapshot_count')):
        prefix='snapshot_%d_'%j;snapshots.append({k[len(prefix):]:v for k,v in arrays.items() if k.startswith(prefix)})
    return dict(meta,order=arrays['order'].tolist(),startup=int(arrays['startup']),snapshots=snapshots)

def get_history(target,seed,reference,ids,X,labels,baseline,packet):
    folder=store()/'histories'/target/str(seed);folder.mkdir(parents=True,exist_ok=True)
    path=folder/'history.npz';receipt=folder/'history.json'
    if receipt.exists():
        meta=load(receipt)
        if sha(path)!=meta['file_sha256']:raise RuntimeError('Saved history checkpoint changed')
        pack=loadz(path)
    elif config()['mode']=='saved_histories':
        prefix='runs/'+history_key(target,seed)+'/'
        checkpoint=packet.json(prefix+'checkpoint.json')
        lr=packet.arrays(prefix+reference['lr_file']);svm=packet.arrays(prefix+'svm_3000.npz')
        if int(lr['count'])!=3000:raise RuntimeError('Saved endpoint differs')
        pack={'order':np.asarray(checkpoint['order'],dtype=np.int64),'startup':np.asarray(checkpoint['startup']),
              'lr':lr['probability'],'svm':svm['scores']}
        meta={'scope':'Reused original purchased IDs and model scores; no refitting'}
    else:
        from learning import acquire,fit_svm
        truth=dict(zip(ids,labels.tolist()));oracle=core.PaidOracle(truth,3000)
        state=acquire(ids,X,oracle,seed,checkpoint=lambda s:save_progress(folder,s),resume=read_progress(folder))
        if state['status']!='CANDIDATE':raise RuntimeError('Rebuild did not reproduce the originally available candidate')
        arrays,svm_meta=fit_svm(X,ids,oracle.observed,seed)
        pack={'order':np.asarray(state['order'],dtype=np.int64),'startup':np.asarray(state['startup']),
              'lr':state['snapshots'][-1]['probability'],'svm':arrays['scores']}
        meta={'scope':'Classifiers refitted from cached original embeddings',
              'lr_fits':state['fits'],'svm_fit':svm_meta,'runtime':runtime()}
    verify_history(pack,reference,ids,labels,baseline)
    if not receipt.exists():
        savez(path,**pack);meta['file_sha256']=sha(path);dump(receipt,meta)
    return pack

def compare_baseline(row,reference):
    numeric=['conditional_accept_probability','correct_accept_probability','false_accept_probability',
             'expected_audit_inspected','expected_total_attention','evaluator_only_recall']
    exact=['target','seed','workflow','training_count','startup_paid','flagged_count','overlap','pre_attention',
           'flagged_id_hash','purchased_history_hash','design','design_status','evaluator_only_M','evaluator_only_T',
           'evaluator_only_b','evaluator_only_adequate','planned_total_attention','realized_total_attention',
           'realized_accepted','terminal_reason']
    for key in exact:
        if row[key]!=reference[key]:raise RuntimeError('25% baseline mismatch: '+key)
    for key in numeric:
        if abs(row[key]-reference[key])>1e-10:raise RuntimeError('25% baseline numerical mismatch: '+key)

def check_baseline_gate():
    receipt=load(store()/'BASELINE_VERIFIED.json')
    assert receipt['experiment_sha256']==sha(ROOT/'experiment.json') and receipt['project_sha256']==sha(ROOT/'project.json')
    for relative,digest in receipt['files'].items():
        if sha(store()/relative)!=digest:raise RuntimeError('Verified baseline or history changed: '+relative)
    assert receipt['rows']==400 and receipt['histories']==100
    return receipt

def main(phase):
    project_hash=check_sources();r=validate_runtime();settings=config();exp=load(ROOT/'experiment.json')
    ref=load(ROOT/'reference_25.json');assert sha(ROOT/'reference_25.json')==exp['reference_25_sha256']
    ids,X,all_labels,codes=verified_arrays()
    folder=store();folder.mkdir(parents=True,exist_ok=True)
    freeze={'experiment_sha256':sha(ROOT/'experiment.json'),'project_sha256':project_hash,
            'reference_sha256':sha(ROOT/'reference_25.json'),'input_sha256':ref['input_sha256'],'mode':settings['mode'],'runtime':r}
    frozen=folder/'EXECUTION.json'
    if frozen.exists() and load(frozen)!=freeze:raise RuntimeError('Source, mode, inputs or runtime changed; use a separate output folder')
    if not frozen.exists():dump(frozen,freeze)
    if phase=='sensitivity':check_baseline_gate()
    percent=25 if phase=='baseline' else 30;cap=len(ids)*percent//100
    needs_packet=any(not (folder/'histories'/h['target']/str(h['seed'])/'history.json').exists() for h in ref['histories'])
    packet=find_packet(required=True) if settings['mode']=='saved_histories' and phase=='baseline' and needs_packet else None
    lookup={(r['target'],r['seed'],r['workflow']):r for r in ref['baseline_rows']}
    records=[];traces=[]
    for index,h in enumerate(ref['histories']):
        target,seed=h['target'],h['seed'];labels=all_labels[:,codes.index(target)].astype(np.int8)
        baseline=[lookup[(target,seed,w['id'])] for w in exp['workflows']]
        if phase=='baseline':pack=get_history(target,seed,h,ids,X,labels,baseline,packet)
        else:
            pack=loadz(folder/'histories'/target/str(seed)/'history.npz')
            verify_history(pack,h,ids,labels,baseline)
        truth=dict(zip(ids,labels.tolist()));order=pack['order'].tolist();purchased=[ids[i] for i in order]
        audit_order=core.common_order(ids,seed,exp['audit_seed_base'])
        cache=folder/'evaluations'/str(percent)/target/str(seed);marker=cache/'COMPLETE.json'
        if marker.exists():
            for name,digest in load(marker)['files'].items():
                if sha(cache/name)!=digest:raise RuntimeError('Evaluation checkpoint changed')
            batch=load(cache/'rows.json');batch_traces=load(cache/'traces.json')
            assert len(batch)==len(batch_traces)==4
        else:
            batch=[];batch_traces=[]
            for workflow in exp['workflows']:
                row,trace,flags=evaluate_candidate(ids,truth,order,int(pack['startup']),pack['lr'],pack['svm'],workflow,seed,'fixed_horizon',cap=cap,flagged_count=exp['flagged_count'])
                row.update(target=target,budget_percent=percent);trace.update(target=target,budget_percent=percent)
                check(row,trace,ids,truth,purchased,flags,audit_order)
                if phase=='baseline':compare_baseline(row,lookup[(target,seed,workflow['id'])])
                batch.append(row);batch_traces.append(trace)
            dump(cache/'rows.json',batch);dump(cache/'traces.json',batch_traces)
            dump(marker,{'files':{n:sha(cache/n) for n in ['rows.json','traces.json']},'independent_checks':'PASS'})
        for row in batch:
            if phase=='baseline':compare_baseline(row,lookup[(target,seed,row['workflow'])])
        records.extend(batch);traces.extend(batch_traces)
        print('BASELINE_VERIFIED' if phase=='baseline' else 'BUDGET_EVALUATED',index+1,'/ 100',target,seed,'budget',percent,flush=True)
    if packet:packet.close()
    assert len(records)==400 and len({(r['target'],r['seed'],r['workflow']) for r in records})==400
    dump(folder/('rows_%d.json'%percent),records);dump(folder/('traces_%d.json'%percent),traces)
    if phase=='baseline':
        paths=[folder/'rows_25.json',folder/'traces_25.json']
        paths+=sorted((folder/'histories').glob('*/*/history.npz'))
        assert len(paths)==102
        dump(folder/'BASELINE_VERIFIED.json',{'experiment_sha256':sha(ROOT/'experiment.json'),'project_sha256':project_hash,
             'rows':400,'histories':100,'files':{str(p.relative_to(folder)):sha(p) for p in paths},
             'scope':'All purchased-history hashes, full final score hashes, flagged IDs, 25% audit designs, metrics and traces reconciled before sensitivity evaluation'})
    else:
        dump(folder/'SENSITIVITY_VERIFIED.json',{'rows':400,'budgets':[25,30],'checks':'PASS',
             'baseline_sha256':sha(folder/'BASELINE_VERIFIED.json'),
             'files':{n:sha(folder/n) for n in ['rows_30.json','traces_30.json']}})
    print('PHASE_COMPLETE',phase,flush=True)

if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('--phase',choices=['baseline','sensitivity'],required=True)
    main(parser.parse_args().phase)


## 5. Define the analysis before execution

LR is primary. Report its 30% acceptance estimate, whether it exceeds 80%, and separately whether its approximate 95% interval lower endpoint exceeds 80%. Report paired acceptance and expected-cost changes, all other rankers, and every species. The target-stratified bootstrap preserves pairing across budgets and methods. It conditions on these five species and this fixed pool.


In [ ]:
%%writefile analyze_budget.py
"""Paired budget sensitivity conditional on the five fixed target tasks."""
from support import *
from collections import Counter
import csv

def calculate(rows,exp,indices):
    lookup={(r['budget_percent'],r['target'],r['seed'],r['workflow']):r for r in rows}
    assert len(rows)==len(lookup)==800
    summaries=[];contrasts=[];by_target=[]
    def values(percent,workflow,key):
        return np.asarray([[lookup[(percent,t['code'],seed,workflow)][key] for seed in t['seeds']] for t in exp['targets']],dtype=float)
    def summarize(v):
        resampled=np.stack([v[i][indices[i]].mean(axis=1) for i in range(5)]).mean(axis=0)
        return float(v.mean()),np.percentile(resampled,[2.5,97.5],method='linear').tolist()
    for w in exp['workflows']:
        name=w['id']
        for budget in exp['budgets_percent']:
            y=values(budget,name,'correct_accept_probability');cost=values(budget,name,'expected_total_attention')
            mean_y,ci_y=summarize(y);mean_c,ci_c=summarize(cost)
            summaries.append({'workflow':name,'budget_percent':budget,'mean_acceptance':mean_y,'acceptance_interval_95':ci_y,
              'mean_expected_windows':mean_c,'expected_windows_interval_95':ci_c,
              'point_estimate_above_80':bool(mean_y>exp['acceptance_benchmark']),
              'interval_lower_endpoint_above_80':bool(ci_y[0]>exp['acceptance_benchmark'])})
            for i,t in enumerate(exp['targets']):
                selected=[lookup[(budget,t['code'],seed,name)] for seed in t['seeds']]
                by_target.append({'target':t['code'],'common_name':t['common_name'],'workflow':name,'budget_percent':budget,
                 'mean_acceptance':float(y[i].mean()),'acceptance_interval_95':np.percentile(y[i][indices[i]].mean(axis=1),[2.5,97.5],method='linear').tolist(),
                 'mean_expected_windows':float(cost[i].mean()),
                 'mean_recall':sum(r['evaluator_only_recall'] for r in selected)/20,
                 'audit_statuses':dict(Counter(r['design_status'] for r in selected)),
                 'cutoffs':dict(Counter(str(r['design']['c']) for r in selected if r['design_status']=='AUDIT'))})
        delta=values(30,name,'correct_accept_probability')-values(25,name,'correct_accept_probability')
        dc=values(30,name,'expected_total_attention')-values(25,name,'expected_total_attention')
        my,cy=summarize(delta);mc,cc=summarize(dc)
        contrasts.append({'workflow':name,'comparison':'30% minus 25%','mean_acceptance_difference':my,'difference_interval_95':cy,
                          'mean_expected_windows_difference':mc,'windows_difference_interval_95':cc,
                          'improved_histories':int((delta>1e-10).sum()),'unchanged_histories':int((np.abs(delta)<=1e-10).sum()),
                          'worsened_histories':int((delta< -1e-10).sum())})
    return summaries,contrasts,by_target

def main():
    check_sources();validate_runtime();folder=store();exp=load(ROOT/'experiment.json')
    from budget_experiment import check_baseline_gate
    check_baseline_gate();sensitivity=load(folder/'SENSITIVITY_VERIFIED.json')
    assert sensitivity['baseline_sha256']==sha(folder/'BASELINE_VERIFIED.json')
    for name,digest in sensitivity['files'].items():assert sha(folder/name)==digest
    rows=load(folder/'rows_25.json')+load(folder/'rows_30.json')
    rng=np.random.Generator(np.random.PCG64(exp['analysis']['bootstrap_seed']))
    indices=np.asarray([rng.integers(0,20,(exp['analysis']['resamples'],20)) for _ in exp['targets']],dtype=np.int64)
    summaries,contrasts,by_target=calculate(rows,exp,indices)
    output=folder/'results';output.mkdir(parents=True,exist_ok=True)
    dump(output/'summary.json',summaries);dump(output/'contrasts.json',contrasts);dump(output/'by_target.json',by_target)
    savez(output/'bootstrap_indices.npz',indices=indices)
    primary=next(r for r in summaries if r['workflow']==exp['primary_workflow'] and r['budget_percent']==30)
    decision={'analysis':'Post hoc budget sensitivity; original 25% comparison retained',
       'primary_workflow':exp['primary_workflow'],'benchmark':0.8,**primary,
       'bootstrap_seed':exp['analysis']['bootstrap_seed'],'bootstrap_indices_sha256':array_hash(indices),
       'interpretation':'Point estimate and interval criterion are distinct. Intervals condition on these targets and this fixed recording pool.',
       'scope':load(folder/'EXECUTION.json')['mode'],'new_audio_or_embeddings':False}
    dump(output/'decision.json',decision)
    with (output/'history_results.csv').open('w',newline='') as f:
        keys=['target','seed','workflow','budget_percent','attention_cap','training_count','overlap','evaluator_only_recall',
              'design_status','correct_accept_probability','expected_total_attention','planned_total_attention','realized_total_attention','realized_accepted']
        writer=csv.DictWriter(f,fieldnames=keys,extrasaction='ignore');writer.writeheader();writer.writerows(rows)
    names={'fixed_lr':'LR','fixed_svm':'SVM','fixed_equal_blend':'Equal rank blend','fixed_3to1_blend':'3:1 LR/SVM rank blend'}
    lines=['# Sapsucker Woods: budget sensitivity','','Kenneth Mwangi Kabiru — University of Leeds','',
      'Post hoc analysis of the same selected pool and acquisition histories. The original 25% result is unchanged.',
      'Training remains at 3,000 labels; the flagged set remains 2,205 windows. Recall requirement: 90%; single-attempt false-acceptance bound: 5%.',
      'The existing allocator is used at both budgets. No policy chooses designs from hidden outcomes or retains designs based on observed audit success.','',
      '| Method | Budget | Correct acceptance (95% interval) | Expected windows |', '|---|---:|---:|---:|']
    for row in summaries:
        lo,hi=row['acceptance_interval_95']
        lines.append('| %s | %d%% | %.2f%% [%.2f, %.2f] | %.2f |'%(names[row['workflow']],row['budget_percent'],100*row['mean_acceptance'],100*lo,100*hi,row['mean_expected_windows']))
    lines+=['','The 30%% LR point estimate is %s 80%%. Its approximate 95%% interval lower endpoint is %s 80%%.'%
            ('above' if primary['point_estimate_above_80'] else 'not above','above' if primary['interval_lower_endpoint_above_80'] else 'not above'),
      'This is an average across five fixed targets, not a guarantee for every species or for new recordings.','',
      '| Method | Acceptance change (pp; 95% interval) | Expected-window change | Improved / tied / worsened histories |', '|---|---:|---:|---:|']
    for row in contrasts:
        lo,hi=row['difference_interval_95']
        lines.append('| %s | %+.3f [%+.3f, %+.3f] | %+.2f | %d / %d / %d |'%(names[row['workflow']],100*row['mean_acceptance_difference'],100*lo,100*hi,row['mean_expected_windows_difference'],row['improved_histories'],row['unchanged_histories'],row['worsened_histories']))
    lines+=['','| Target | Method | 25% acceptance | 30% acceptance |','|---|---|---:|---:|']
    for target in exp['targets']:
        for workflow in exp['workflows']:
            pair=[next(r for r in by_target if r['target']==target['code'] and r['workflow']==workflow['id'] and r['budget_percent']==b) for b in [25,30]]
            lines.append('| %s | %s | %.2f%% | %.2f%% |'%(target['common_name'],names[workflow['id']],100*pair[0]['mean_acceptance'],100*pair[1]['mean_acceptance']))
    lines+=['','Correct acceptance is the mean calculated audit acceptance probability multiplied by retrospective candidate adequacy. Sampled audit verdicts are retained separately.',
      'The budget ceiling rises from 5,513 to 6,615 unique labels (1,102 additional; about 20% more than the original ceiling). Expected expenditure includes non-acceptance and early audit failure.',
      'All 100 histories enter each method average. Secondary intervals are descriptive, without a family-wide superiority claim.',
      'The benchmark concerns these observed classifier qualities. It does not establish 80% audit power for every classifier with 95% recall, a new-species guarantee, or a successful adaptive training-stopping controller.',
      'Reproduction mode: '+config()['mode']+'. Embeddings were verified and reused; no audio retrieval or embedding inference occurred.','']
    (output/'REPORT.md').write_text('\n'.join(lines))
    print('PAIRED_ANALYSIS_COMPLETE; run the independent statistics check before exporting',flush=True)

if __name__=='__main__':main()


## 6. Install the isolated CPU runtime and verify inputs

This uses the original Python and numerical-library versions without invoking Colab's system `venv`. Dependency downloads may be needed. No audio or embedding download can be triggered. A complete cache or original results packet is required; missing or mismatched inputs stop the notebook.


In [ ]:
for name,expected in spec['execution_files'].items():
    if digest(RUN_ROOT/name)!=expected:raise RuntimeError('Executed scientific source differs from the release: '+name)
RUNTIMES=RUN_ROOT.parent.parent/'runtimes'
subprocess.run([sys.executable,str(RUN_ROOT/'environment.py'),'--directory',str(RUNTIMES)],check=True)
PYTHON=str(RUNTIMES/'analysis/bin/python')
def run(script,*arguments):
    env=dict(os.environ,PYTHONUNBUFFERED='1',PYTHONDONTWRITEBYTECODE='1')
    process=subprocess.Popen([PYTHON,str(RUN_ROOT/script),*map(str,arguments)],cwd=RUN_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:print(line,end='',flush=True)
        if process.wait():raise RuntimeError(script+' failed; inspect its output. Completed verified checkpoints are retained.')
    finally:
        if process.poll() is None:process.terminate();process.wait()
run('inputs.py')
EXPERIMENT=json.loads((RUN_ROOT/'experiment.json').read_text())
STORE=OUTPUT/digest(RUN_ROOT/'experiment.json')[:16]/RUN_MODE
print('Original pool:',EXPERIMENT['N'],'windows. Budgets:',EXPERIMENT['N']//4,'and',EXPERIMENT['N']*3//10)
print('Mode:',RUN_MODE)


## 7. Reproduce the complete 25% baseline

This stage reuses scores or refits from cached embeddings according to `RUN_MODE`. It compares hashes and every design, probability, cost and verdict against the labelled 25% reference outputs. All 100 histories must pass. A refit mismatch is reported as a reproduction failure, not silently treated as another history.

Rerunning resumes verified checkpoints. Do not run multiple Colab sessions against the same output folder at once.


In [ ]:
run('budget_experiment.py','--phase','baseline')
print((STORE/'BASELINE_VERIFIED.json').read_text()[:700])


## 8. Evaluate the 30% budget

This is the new sensitivity calculation. It changes only the evaluation allowance to 6,615 unique labels. It uses the existing allocator at the larger cap; it does not choose an audit using hidden outcomes or preserve a design because its realised audit passed. Each history's exact risk boundaries, adjacent choices, expected cost and paid trace are independently checked.


In [ ]:
run('budget_experiment.py','--phase','sensitivity')


## 9. Calculate and independently verify the paired results

The saved rows determine the answer; the earlier Towhee projection is not inserted as a result. Point estimates above 80% and interval lower bounds above 80% are separate statements. Overall success does not mean every species exceeds 80%.


In [ ]:
run('analyze_budget.py')
run('recheck_statistics.py')
from IPython.display import display, Markdown
display(Markdown((STORE/'results/REPORT.md').read_text()))


## 10. Inspect acceptance and cost

The dashed line marks the 80% acceptance benchmark. Bars show the separate budget results, with uncertainty intervals in the report. The plot uses the saved analysis outputs; it does not refit or select methods.


In [ ]:
import matplotlib.pyplot as plt
summary=json.loads((STORE/'results/summary.json').read_text())
methods=['fixed_lr','fixed_svm','fixed_equal_blend','fixed_3to1_blend']
labels=['LR','SVM','Equal blend','3:1 blend']
fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for budget,offset,color in [(25,-.18,'#526D82'),(30,.18,'#258579')]:
    selected=[next(r for r in summary if r['workflow']==w and r['budget_percent']==budget) for w in methods]
    x=[i+offset for i in range(4)]
    axes[0].bar(x,[100*r['mean_acceptance'] for r in selected],width=.35,label=str(budget)+'% budget',color=color)
    axes[1].bar(x,[r['mean_expected_windows'] for r in selected],width=.35,label=str(budget)+'% budget',color=color)
axes[0].axhline(80,color='#9B3D33',linestyle='--',linewidth=1.3,label='80% benchmark')
axes[0].set(ylabel='Mean correct acceptance (%)',ylim=(0,105))
axes[1].set(ylabel='Expected unique windows reviewed')
for ax in axes:
    ax.set_xticks(range(4),labels);ax.spines[['top','right']].set_visible(False);ax.legend(fontsize=8)
plt.show()
fig.savefig(STORE/'results/budget_comparison.png',dpi=180)


## 11. Export scientific results to Drive

The export contains source, settings, input hashes, target labels, final history arrays, audit rows/traces and verified summaries. It excludes local configuration paths, package installation logs, acquisition debugging records and notebook outputs. The original embedding cache and 25% study are untouched. This is a separate post hoc analysis, not a replacement result or a new independent dataset.


In [ ]:
run('export_results.py')
receipt=json.loads((STORE/'EXPORT.json').read_text())
print('Saved:',STORE/'exports'/receipt['filename'])
print('SHA-256:',receipt['sha256'])
